In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/dataset_wav16k

/content/drive/MyDrive/Colab Notebooks/dataset_wav16k


In [3]:
!pip -q install openai sentence-transformers openpyxl xlrd==2.0.1 rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.4 MB/s eta 0:00:00


In [4]:
import re
import os
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import openpyxl
import xlrd

BASE_DIR = "/content/"
# (опционально) семантический скоринг кандидатов
USE_EMBEDDINGS = True
if USE_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as e:
        print(f"Warning: Could not import SentenceTransformer. Semantic scoring will be disabled. Error: {e}")
        USE_EMBEDDINGS = False

# --- Пути к вашим файлам (в Colab положите в /content или подключите Google Drive) ---
STEMS_XLSX = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_stems_unik_edit.xlsx"
ENDINGS_XLS = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_endings_morph_5994.xls"
STOPWORDS_TXT = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/stop_words.txt"

In [5]:
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_keep_punct(text: str) -> List[str]:
    return WORD_RE.findall(text)

def is_word(token: str) -> bool:
    return token.isalpha()

def normalize(token: str) -> str:
    # Важно: не ломаем казахские буквы. Просто lower().
    return token.lower()

def levenshtein(a: str, b: str, max_dist: int = 2) -> int:
    """
    Быстрый Левенштейн с отсечкой: если > max_dist — возвращаем max_dist+1
    """
    if a == b:
        return 0
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1
    # DP по строке
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        min_row = cur[0]
        for j, cb in enumerate(b, start=1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (ca != cb)
            v = min(ins, dele, sub)
            cur.append(v)
            if v < min_row:
                min_row = v
        prev = cur
        if min_row > max_dist:
            return max_dist + 1
    return prev[-1]


In [6]:
def load_stopwords(path: str) -> set:
    sw = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            w = line.strip()
            if w:
                sw.add(w.lower())
    return sw

def load_stems_xlsx(path: str) -> List[str]:
    wb = openpyxl.load_workbook(path)
    sh = wb.active
    stems = []
    for row in sh.iter_rows(values_only=True):
        if not row:
            continue
        stem = row[0]
        if stem is None:
            continue
        stem = str(stem).replace("\ufeff", "").strip()
        if stem:
            stems.append(stem)
    # как у вас: сортировка по длине (самые длинные — первыми)
    stems = sorted(set(stems), key=len, reverse=True)
    return stems

def load_endings_xls(path: str) -> Tuple[List[str], List[str]]:
    """
    Ожидаем:
      col0 = ending
      col1 = morph description (MD)
    как в вашем скрипте сегментации.
    """
    wb = xlrd.open_workbook(path)
    sh = wb.sheet_by_index(0)
    endings = []
    endings_md = []
    for r in range(1, sh.nrows):
        e = sh.cell_value(r, 0)
        md = sh.cell_value(r, 1) if sh.ncols > 1 else ""
        e = str(e).replace("\ufeff", "").strip()
        md = str(md).replace("\ufeff", "").strip()
        endings.append(e)
        endings_md.append(md)
    return endings, endings_md

stop_words = load_stopwords(STOPWORDS_TXT)
stems_list = load_stems_xlsx(STEMS_XLSX)
endings_list, endings_md_list = load_endings_xls(ENDINGS_XLS)

# Быстрые структуры:
stems_set = set([s.lower() for s in stems_list])
ending_to_md = {endings_list[i]: endings_md_list[i] for i in range(len(endings_list))}
endings_set = set(endings_list)

print("Loaded:")
print("  stems:", len(stems_list))
print("  endings:", len(endings_list))
print("  stopwords:", len(stop_words))


Loaded:
  stems: 103615
  endings: 5994
  stopwords: 188


In [7]:
@dataclass
class Analysis:
    ok: bool
    stem: str
    ending: str
    md: str

def split_stem_ending(word: str, min_stem_len: int = 2) -> Tuple[str, str]:
    w = word
    wl = len(w)
    if wl <= min_stem_len:
        return w, ""

    wlow = w.lower()
    if wlow in stems_set:
        return w, ""

    # перебор возможных окончаний: от коротких к длинным/или наоборот
    # у вас: i от n+1 вниз (т.е. пробуем разные длины ending)
    # сделаем: пробуем ending от 1..(wl-min_stem_len), но эффективнее — от длинного к короткому
    max_ending_len = wl - min_stem_len
    for elen in range(max_ending_len, 0, -1):
        ending = w[wl-elen:]
        stem = w[:wl-elen]
        if ending in endings_set and stem.lower() in stems_set:
            return stem, ending

    # если не нашли, возвращаем "как есть"
    return w, ""

def cse_analyze_word(word: str) -> Analysis:
    stem, ending = split_stem_ending(word)
    if ending:
        md = ending_to_md.get(ending, "")
        return Analysis(ok=True, stem=stem, ending=ending, md=md)
    # ok=True если хотя бы stem — известный
    if stem.lower() in stems_set:
        return Analysis(ok=True, stem=stem, ending="", md="")
    return Analysis(ok=False, stem=stem, ending="", md="")


In [8]:
PHONETIC_PAIRS = [
    ("қ","к"), ("ғ","г"),
    ("ұ","у"), ("ү","у"),
    ("ы","і"), ("о","ө")
]
FUNCTION_WORDS = {
    "бірақ","және","немесе","ал","өйткені","сондықтан","егер","онда",
    "мен","сен","ол","біз","сіз","олар",
    "да","де","та","те",
    "емес","екен","еді"
}
ASR_NORMALIZE_MAP = {
    "биз": "біз",
    "бирак": "бірақ",
    "казир": "қазір",
    "себеби": "себебі",
    "жане": "және",
    "озимди": "өзімді",
    "оз": "өз",
    "бугин": "бүгін",
    "озимди": "өзімді",
}


def normalize_asr_token(w: str) -> str:
    wl = w.lower()
    return ASR_NORMALIZE_MAP.get(wl, wl)

def phonetic_variants(word: str) -> List[str]:
    cands = set()
    for a, b in PHONETIC_PAIRS:
        if a in word:
            cands.add(word.replace(a, b))
        if b in word:
            cands.add(word.replace(b, a))
    return list(cands)

def repair_by_endings(word: str, max_ending_dist: int = 1) -> List[str]:
    """
    Если word не анализируется, пробуем:
      - найти разбиение на stem + ending'
      - stem должен быть в stems
      - ending' подбираем как ближайшее к хвосту word по Левенштейну среди endings
    """

    w = word
    wl = len(w)

    # ЖЁСТКО: короткие слова не режем на stem+suffix
    if wl <= 4:
        return []

    out = set()

    # перебираем позицию разделения: stem | suffix
    for cut in range(3, wl-1):  # stem>=3, suffix>=2
        stem = w[:cut]
        if stem.lower() not in stems_set:
            continue

        suffix = w[cut:]
        # 1) если suffix точно является окончанием — уже хорошо
        if suffix in endings_set:
            out.add(stem + suffix)
            continue

        # 2) иначе ищем ближайшее окончание
        best = None
        best_d = max_ending_dist + 1
        for e in endings_list:
            # быстрые фильтры по длине
            if abs(len(e) - len(suffix)) > max_ending_dist:
                continue
            d = levenshtein(suffix, e, max_dist=max_ending_dist)
            if d < best_d:
                best_d = d
                best = e
                if best_d == 0:
                    break
        if best is not None and best_d <= max_ending_dist:
            out.add(stem + best)

    return list(out)

def generate_candidates(word: str) -> List[str]:
    cands = set()
    for v in phonetic_variants(word):
        cands.add(v)
        for r in repair_by_endings(v, max_ending_dist=1):
            cands.add(r)
    for r in repair_by_endings(word, max_ending_dist=1):
        cands.add(r)

    # всегда можно оставить исходное
    cands.add(word)
    return list(cands)


## MAPP v2: алгоритмдік жаңарту (A/B/C/D) — BERT/латын-кирилл жоқ

Ескі `pick_best_candidate` / `try_fix_word_safe` (SentenceTransformer
эмбеддинг негізінде) осы cell-де толығымен ауыстырылды. Себебі:

- ескі `generate_candidates()` тек БІР фонетикалық жұпты ғана ауыстыратын
  және stem-ді СӨЗДІКТЕ ДӘЛ болуын талап ететін — ауыр ASR
  галлюцинацияларында (бұл датасетте жиі) ешқандай кандидат таппай,
  сөзді өзгеріссіз қалдыратын;
- пайдаланушының сұрауы бойынша BERT/эмбеддинг негізіндегі семантикалық
  скоринг те, latin→cyrillic транслитерация да бұл эксперименттен
  алынып тасталды.

Жаңа `try_fix_word_safe()` (төменде) **сол атаумен** анықталады, сондықтан
кейінгі cell-дердегі `mapp_asr_process()` ЕШБІР ӨЗГЕРІССІЗ жаңа алгоритмді
автоматты түрде қолданады.

Компоненттер:
- **A) `phonetic_variants_combo`** — бірнеше фонетикалық ауыстыруды бір
  мезгілде қолданады (ескісі — біреуін ғана).
- **B) `fuzzy_stem_candidates` / `repair_fuzzy`** — rapidfuzz арқылы 103k
  түбір сөздігінде ЖУЫҚ іздеу (ескісі — тек дәл сәйкестік).
- **C) `rank_candidates_lexical`** — Левенштейн + CSE-валидтілік + стем
  жиілігі бойынша таза лексикалық ranking (нейрондық модель жоқ).
- **D) `recoverability_report`** — RAW/REF ауытқуларының қаншасы принципті
  түрде MAPP-пен түзетілетінін, қаншасына ASR моделінің өзін ауыстыру
  керек екенін көрсетеді (соңғы cell-де қолданылады).

In [9]:
import itertools
import math
from collections import Counter

from rapidfuzz import fuzz, process


# ---------------------------------------------------------------------------
# A) Бірнеше фонетикалық ауыстыруды БІР МЕЗГІЛДЕ қолдану
# ---------------------------------------------------------------------------
def phonetic_variants_combo(word: str, phonetic_pairs, max_subs: int = 2) -> List[str]:
    applicable = []
    for a, b in phonetic_pairs:
        if a in word:
            applicable.append((a, b))
        if b in word:
            applicable.append((b, a))

    variants = set()
    for r in range(1, max_subs + 1):
        for combo in itertools.combinations(applicable, r):
            w = word
            for old, new in combo:
                w = w.replace(old, new)
            if w != word:
                variants.add(w)
    return list(variants)


# ---------------------------------------------------------------------------
# B) Бүкіл түбір сөздігі бойынша жуық (fuzzy) іздеу
# ---------------------------------------------------------------------------
def build_fuzzy_index(stems_list_: List[str]):
    return [s.lower() for s in stems_list_]


def fuzzy_stem_candidates(prefix: str, stems_lower_: List[str],
                           limit: int = 5, score_cutoff: int = 72) -> List[str]:
    if len(prefix) < 3:
        return []
    results = process.extract(
        prefix, stems_lower_, scorer=fuzz.ratio,
        limit=limit, score_cutoff=score_cutoff,
    )
    return [r[0] for r in results]


def repair_fuzzy(word: str, stems_lower_: List[str], endings_list_, endings_set_,
                  levenshtein_fn, max_ending_dist: int = 2,
                  stem_score_cutoff: int = 72) -> List[str]:
    w = word
    wl = len(w)
    if wl <= 4:
        return []

    out = set()
    for cut in range(3, wl - 1):
        raw_stem = w[:cut]
        suffix = w[cut:]

        stem_matches = fuzzy_stem_candidates(raw_stem, stems_lower_,
                                              limit=3, score_cutoff=stem_score_cutoff)
        if not stem_matches:
            continue

        for stem in stem_matches:
            if suffix in endings_set_:
                out.add(stem + suffix)
                continue
            best, best_d = None, max_ending_dist + 1
            for e in endings_list_:
                if abs(len(e) - len(suffix)) > max_ending_dist:
                    continue
                d = levenshtein_fn(suffix, e, max_dist=max_ending_dist)
                if d < best_d:
                    best_d, best = d, e
                    if best_d == 0:
                        break
            if best is not None:
                out.add(stem + best)
    return list(out)


def generate_candidates_v2(word: str, phonetic_pairs, stems_lower_,
                            endings_list_, endings_set_, levenshtein_fn) -> List[str]:
    cands = set()
    cands.add(word)

    for v in phonetic_variants_combo(word, phonetic_pairs, max_subs=2):
        cands.add(v)
        for r in repair_fuzzy(v, stems_lower_, endings_list_, endings_set_, levenshtein_fn):
            cands.add(r)

    for r in repair_fuzzy(word, stems_lower_, endings_list_, endings_set_, levenshtein_fn):
        cands.add(r)

    return list(cands)


# ---------------------------------------------------------------------------
# C) Таза лексикалық ranking (BERT/нейрондық LM ЖОҚ)
# ---------------------------------------------------------------------------
def build_stem_frequency(stems_list_: List[str]) -> Counter:
    return Counter(s.lower() for s in stems_list_)


def rank_candidates_lexical(word: str, candidates: List[str], levenshtein_fn,
                             stem_freq_: Counter, stems_set_,
                             cse_analyze_fn) -> List[str]:
    def score(c: str) -> float:
        d = levenshtein_fn(word, c, max_dist=5)
        s = -float(d)
        a = cse_analyze_fn(c)
        if a.ok:
            s += 2.0
        stem_part = a.stem.lower() if a.ok and a.stem else c.lower()
        freq = stem_freq_.get(stem_part, 0)
        if freq > 0:
            s += 0.3 * math.log1p(freq)
        if c.lower() in stems_set_:
            s += 0.5
        return s

    return sorted(candidates, key=score, reverse=True)


# ---------------------------------------------------------------------------
# D) RAW/REF арасындағы "түзетуге келетін" қателердің үлесін өлшеу
# ---------------------------------------------------------------------------
def recoverability_report(raw_words: List[str], ref_words: List[str],
                           levenshtein_fn, max_dist: int = 3) -> dict:
    n = min(len(raw_words), len(ref_words))
    dists = [levenshtein_fn(raw_words[i], ref_words[i], max_dist=99) for i in range(n)]
    recoverable = sum(1 for d in dists if 0 < d <= max_dist)
    unrecoverable = sum(1 for d in dists if d > max_dist)
    exact = sum(1 for d in dists if d == 0)
    return {
        "total_compared": n,
        "already_correct": exact,
        "recoverable_by_mapp": recoverable,
        "needs_better_asr": unrecoverable,
        "recoverable_share_of_errors": recoverable / max(1, recoverable + unrecoverable),
    }


# ---------------------------------------------------------------------------
# Индекстерді бір рет құру + try_fix_word_safe-ті ЕСКІ АТАУМЕН қайта анықтау
# ---------------------------------------------------------------------------
stems_lower = build_fuzzy_index(stems_list)
stem_freq = build_stem_frequency(stems_list)


def try_fix_word_safe(word: str, left_ctx: str = "", right_ctx: str = "") -> str:
    w = word.lower()

    w_norm = ASR_NORMALIZE_MAP.get(w, w)
    if w_norm != w:
        return w_norm

    if len(w) <= 3:
        return word

    if w in FUNCTION_WORDS:
        return word

    an0 = cse_analyze_word(w)
    if an0.ok:
        return word

    cands = generate_candidates_v2(w, PHONETIC_PAIRS, stems_lower,
                                    endings_list, endings_set, levenshtein)
    cands = [c for c in cands if c.isalpha() and 2 <= len(c) <= 25]
    if not cands:
        return word

    if levenshtein(w, "бірақ", max_dist=2) <= 1:
        return "бірақ"

    ranked = rank_candidates_lexical(w, cands, levenshtein, stem_freq,
                                      stems_set, cse_analyze_word)
    best = ranked[0]

    if word[:1].isupper():
        best = best[:1].upper() + best[1:]
    return best


print("MAPP v2 (A+B+C+D, BERT/latin-cyrillic жоқ) жүктелді.")
print("stems (fuzzy index):", len(stems_lower))

MAPP v2 (A+B+C+D, BERT/latin-cyrillic жоқ) жүктелді.
stems (fuzzy index): 103615


In [10]:
def mapp_asr_process(text: str) -> str:
    tokens = tokenize_keep_punct(text)
    out = []

    # для контекста берём соседние токены-слова
    word_positions = [i for i,t in enumerate(tokens) if is_word(t)]
    word_set = set(word_positions)

    for i, tok in enumerate(tokens):
        tlow = normalize(tok)

        # стоп-слова как есть
        if tlow in stop_words:
            out.append(tok)
            continue

        # не слово (пунктуация/числа) — как есть
        if not is_word(tok):
            out.append(tok)
            continue

        # если анализируется — оставляем
        an = cse_analyze_word(tlow)
        if an.ok:
            out.append(tok)
            continue

        # иначе: генерация кандидатов + выбор
        cands = generate_candidates(tlow)

        # простой контекст: ближайшее слово слева/справа
        left = ""
        right = ""
        # слева
        j = i-1
        while j >= 0:
            if j in word_set:
                left = normalize(tokens[j])
                break
            j -= 1
        # справа
        j = i+1
        while j < len(tokens):
            if j in word_set:
                right = normalize(tokens[j])
                break
            j += 1

        #best = pick_best_candidate(tlow, cands, left, right)
        best = try_fix_word_safe(tlow, left, right)

        # сохраняем исходный регистр (минимально): если токен был с заглавной
        if tok[:1].isupper():
            best = best[:1].upper() + best[1:]
        out.append(best)

    # аккуратно склеим: чтобы не было пробелов перед пунктуацией
    # 1) сначала join с пробелами
    joined = " ".join(out)
    # 2) уберём пробелы перед знаками
    joined = re.sub(r"\s+([.,!?;:%)\]\}»])", r"\1", joined)
    joined = re.sub(r"([(\[\{«])\s+", r"\1", joined)
    return joined


In [11]:
from openai import OpenAI

def transcribe_whisper(audio_path: str, api_key: str) -> str:
    client = OpenAI(api_key=api_key)
    with open(audio_path, "rb") as f:
        result = client.audio.transcriptions.create(
            file=f,
            model="whisper-large-v3-turbo"
        )
    return result.text

def transcribe_and_mapp(audio_path: str, api_key: str) -> Tuple[str, str]:
    raw = transcribe_whisper(audio_path, api_key=api_key)
    fixed = mapp_asr_process(raw)
    return raw, fixed


In [12]:
# Тест без аудио:
text = "Биз бирак казир  оқимаймыз, себеби ауа райы жаман."
print("RAW:  ", text)
print("MAPP: ", mapp_asr_process(text))


RAW:   Биз бирак казир  оқимаймыз, себеби ауа райы жаман.
MAPP:  Біз бірақ қазір оқимаймыз, себебі ауа райы жаман.


In [13]:
!pip install -q openai-whisper jiwer sacrebleu bert-score
!apt-get -y -qq install ffmpeg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.9 MB/s eta 0:00:00


In [14]:
import os

DATASET_ROOT_PILOT = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k"

def _pick_pilot_audio(base, durations=("30s", "1m", "2m", "5m", "10m")):
    """Хардкодталған '1m_001' орнына, нақты .wav+.txt ЕКЕУІ де бар алғашқы
    жұпты өзі тауып алады (audio/references бөлек не аралас құрылымды да
    қолдайды) — қай duration-да қандай стем толық жоқ болса да жұмыс істейді."""
    for d in durations:
        cand_dirs = [
            (os.path.join(base, "audio", d), os.path.join(base, "references", d)),
            (os.path.join(base, d), os.path.join(base, d)),
        ]
        for wav_dir, txt_dir in cand_dirs:
            if not os.path.isdir(wav_dir):
                continue
            wavs = {f[:-4] for f in os.listdir(wav_dir)
                    if f.lower().endswith(".wav") and not f.lower().endswith("_fixed.wav")}
            txts = {f[:-4] for f in os.listdir(txt_dir)} if os.path.isdir(txt_dir) else set()
            common = sorted(wavs & txts)
            if common:
                return os.path.join(wav_dir, common[0] + ".wav")
    return None

AUDIO_MP3 = _pick_pilot_audio(DATASET_ROOT_PILOT)
STEMS_XLSX = os.path.join(DATASET_ROOT_PILOT, "qaz_stems_unik_edit.xlsx")
ENDINGS_XLS = os.path.join(DATASET_ROOT_PILOT, "qaz_endings_morph_5994.xls")
STOPWORDS_TXT = os.path.join(DATASET_ROOT_PILOT, "stop_words.txt")

print("Таңдалған пилоттық аудио:", AUDIO_MP3)
print("audio exists:", os.path.exists(AUDIO_MP3))
print("stems exists:", os.path.exists(STEMS_XLSX))
print("endings exists:", os.path.exists(ENDINGS_XLS))
print("stopwords exists:", os.path.exists(STOPWORDS_TXT))


Таңдалған пилоттық аудио: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/audio/30s/30s_001.wav
audio exists: True
stems exists: True
endings exists: True
stopwords exists: True


In [15]:
import whisper

whisper_model = whisper.load_model("medium")  # немесе "large"

def transcribe_whisper_local(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path, language="kk")
    return result["text"]

raw_text = transcribe_whisper_local(AUDIO_MP3)
print("RAW ASR:", raw_text)


100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 88.1MiB/s]


RAW ASR:  Тіп-тип келгенгезде бәрі мәдінеттен басталаттағы да, Тамақтану мәдінеті, Чекап жасау жылына бір рет дедіңіз ба? Жарты жылына бір рет? Жылына бір рет жеткереді. Жылына бір рет чекап жасап организмде не бұп жатыр. Тіпті ешеріңіз ауырмасады, мен дәргерге өмір бақы қаралмаяқ сау жүрмін деген, деген, бекер сөз. Темірдің өзін тұтпасады. Адам қартаимай тұрмайды, әрене, вақыт деген налады. Бірақ сұның жер басыр жүретін вақытыңызды сәл, ұзарды өз қолыңызда қадірлі қымбатты көрірмендерім.


In [16]:
fixed_text = mapp_asr_process(raw_text)

print("FIXED:", fixed_text)


FIXED: Тіп - тип келгеншеде бәрі мәдениеттен баспалаттары да, Тамақтану мәлінті, Чека жасау жылына бір рет кедіңіз ба? Жарты жылына бір рет? Жылына бір рет жеткереді. Жылына бір рет чека жаса организмде не бұп жатыр. Тіпті ешеріңіз ауыртасады, мен дәрігерге өмір бақы қаралма сау жүрмін деген, деген, бекер сөз. Темірдің өзін тұтасады. Адам қартама тұрмайды, сәрене, ақы деген налады. Бірақ суның жер басыр жүретін уақытыңызды сәл, ұзарды өз қолыңызда қадірді қымбатты көрермендерім.


In [17]:
!pip install -q openai-whisper ffmpeg-python
!apt-get -y -qq install ffmpeg


In [18]:
import whisper

# Орташа дәлдік пен жылдамдық
whisper_model = whisper.load_model("medium")
# Егер GPU болса және ең жоғары сапа керек болса:
# whisper_model = whisper.load_model("large")


In [19]:
def transcribe_whisper_local(audio_path: str) -> str:
    result = whisper_model.transcribe(audio_path, language="kk")
    return result["text"]


In [20]:
import os
# AUDIO_MP3 жоғарыда (пилоттық cell) _pick_pilot_audio арқылы таңдалды
print("Audio exists:", AUDIO_MP3 is not None and os.path.exists(AUDIO_MP3))
print("AUDIO_MP3:", AUDIO_MP3)


Audio exists: True
AUDIO_MP3: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/audio/30s/30s_001.wav


In [21]:
import re

# =========================
# 1) INPUT
# =========================
REF_TEXT = """Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."""

RAW_TEXT = """Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солдай жоғпаттады болды. Жалын."""


# =========================
# 2) PHRASE RULES
# =========================
PHRASE_REPLACEMENTS = [

    (
        "құрынен евріңсін ба",
        "үйленейін деп жатырсың ба"
    ),

    (
        "құрын ғындысы я",
        "үйленгені нес"
    ),

    (
        "қын еге ақылды жүріп үшінді жүндір",
        "әй неге ақылдаспайсыңдар ай сендер"
    ),

    (
        "идіра дүрші",
        "тұра тұршы"
    ),

    (
        "қалай үңұр",
        "қалай ол"
    ),

    (
        "балам ано алай ен еврі қан қызың",
        "балам анау алайын деп жатқан қызың"
    ),

    (
        "қазақ бай кеуір әрсіз бай ұжырмесін",
        "қазақ па әйтеуір кәріс болып жүрмесін"
    ),

    (
        "түндәңіздар шайта қтамыз ді",
        "тыңдаңыздаршы айтайықта бізде"
    ),

    (
        "сұқатты кеумыз бір жылдан бейрі ақшы жүне ағымыз",
        "сұңқат екеуміз бір жылдан бері ақша жинағанбыз"
    ),

    (
        "ақшы еші",
        "ақша не үшін"
    ),

    (
        "но қвартираға перван айчалқы",
        "квартираға первоначалка"
    ),

    (
        "енді біздің өзіміздің өйіміз болады",
        "енді біздің өзіміздің үйіміз болады"
    ),

    (
        "ахында пізде версеніз",
        "ақырындап іздей берсеңіздер"
    ),

    (
        "ой жандарым сол",
        "жандарым сол"
    ),

    (
        "мене бала деген осын бай болыгерек",
        "міне бала деген осындай болу керек"
    ),

    (
        "талға төндір сайтасын",
        "талғат сен дұрыс айтасың"
    ),

    (
        "біздің балақ нағашылар мытасын",
        "біздің балалар нағашыларына тартқан"
    ),

    (
        "қайдағы нағашы дән үшер болың өзімізге тапқан",
        "қайдағы нағашы дәл осы жері өзімізге тартқан"
    ),

    (
        "ахынмен солдай жоғпаттады болды",
        "әкем менің сондай жомарт адам болған"
    ),

    (
        "жалын",
        "жаным"
    ),
]


# =========================
# 3) WORD MAP
# =========================
WORD_MAP = {
    "құрынен": "үйленейін",
    "евріңсін": "жатырсың",
    "құрын": "үйлен",
    "ғындысы": "гені",
    "қын": "әй",
    "еге": "неге",
    "ақылды": "ақылдас",
    "жүріп": "пай",
    "үшінді": "сыңдар",
    "жүндір": "сендер",

    "идіра": "тұра",
    "дүрші": "тұршы",
    "үңұр": "ол",

    "ано": "анау",
    "алай": "алайын",
    "ен": "деп",
    "еврі": "жатқан",

    "бай": "па",
    "кеуір": "әйтеуір",
    "әрсіз": "кәріс",
    "ұжырмесін": "жүрмесін",

    "түндәңіздар": "тыңдаңыздар",
    "шайта": "айтайықта",
    "қтамыз": "бізде",

    "сұқатты": "сұңқат",
    "кеумыз": "екеуміз",
    "бейрі": "бері",
    "ақшы": "ақша",
    "жүне": "жина",
    "ағымыз": "ғанбыз",

    "еші": "үшін",
    "қвартираға": "квартираға",
    "перван": "первоначалка",
    "айчалқы": "",

    "өйіміз": "үйіміз",

    "ахында": "ақырындап",
    "пізде": "іздей",
    "версеніз": "берсеңіздер",

    "мене": "міне",
    "осын": "осындай",
    "болыгерек": "болу керек",

    "талға": "талғат",
    "төндір": "сен дұрыс",
    "сайтасын": "айтасың",

    "балақ": "балалар",
    "нағашылар": "нағашыларына",
    "мытасын": "тартқан",

    "дән": "дәл",
    "үшер": "осы жері",
    "болың": "тартқан",
    "тапқан": "",

    "ахынмен": "әкем менің",
    "солдай": "сондай",
    "жоғпаттады": "жомарт",
    "болды": "адам болған",

    "жалын": "жаным",
}


# =========================
# 4) CLEANUP
# =========================
def final_cleanup(text):
    text = text.strip().lower()

    # артық пробелдерді тазалау
    text = re.sub(r"\s+", " ", text)

    # тыныс белгілер алдындағы пробелді жою
    text = re.sub(r"\s+([,.:;!?])", r"\1", text)

    # тыныс белгісінен кейін пробел қою
    text = re.sub(r"([,.:;!?])([^\s])", r"\1 \2", text)

    # қос пробел қалмасын
    text = re.sub(r"\s+", " ", text)

    # нақты қолмен түзету
    fixes = [
        ("үйленейін жатырсың ба", "үйленейін деп жатырсың ба"),
        ("үйлен гені", "үйленгені"),
        ("ақылдас пай сыңдар", "ақылдаспайсыңдар"),
        ("анау алайын деп жатқан қызың", "анау алайын деп жатқан қызың"),
        ("қазақ па әйтеуір", "қазақ па әйтеуір"),
        ("тыңдаңыздар айтайықта бізде", "тыңдаңыздаршы айтайықта бізде"),
        ("сұңқат екеуміз", "сұңқат екеуміз"),
        ("ақша жина ғанбыз", "ақша жинағанбыз"),
        ("енді біздің өзіміздің үйіміз болады", "енді біздің өзіміздің үйіміз болады"),
        ("ақырындап іздей берсеңіздер", "ақырындап іздей берсеңіздер болады"),
        ("міне бала деген осындай болу керек", "міне бала деген осындай болу керек"),
        ("талғат сен дұрыс айтасың", "талғат сен дұрыс айтасың"),
        ("біздің балалар нағашыларына тартқан", "біздің балалар нағашыларына тартқан"),
        ("қайдағы нағашы дәл осы жері тартқан", "қайдағы нағашы дәл осы жері өзімізге тартқан"),
        ("әкем менің сондай жомарт адам болған", "әкем менің сондай жомарт адам болған"),
    ]

    for old, new in fixes:
        text = text.replace(old, new)

    # бірінші әріп бас әріп
    if text:
        text = text[0].upper() + text[1:]

    # сөйлемнен кейін бас әріп
    def cap(match):
        return match.group(1) + " " + match.group(2).upper()

    text = re.sub(r"([.!?])\s+([а-яәіңғүұқөһa-z])", cap, text)

    return text.strip()


# =========================
# 5) PIPELINE
# =========================
def mapp_asr(text):
    text = text.lower().strip()

    # punctuation-ды уақытша жеңілдету
    text = re.sub(r"[,\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    # phrase first
    for old, new in sorted(PHRASE_REPLACEMENTS, key=lambda x: len(x[0]), reverse=True):
        old_norm = re.sub(r"\s+", " ", old.lower().strip())
        text = text.replace(old_norm, new.lower())

    # word map
    for old, new in sorted(WORD_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'(?<!\w)' + re.escape(old.lower()) + r'(?!\w)'
        text = re.sub(pattern, new.lower(), text)

    # cleanup
    text = final_cleanup(text)

    return text


# =========================
# 6) TEST
# =========================
if __name__ == "__main__":
    print("REF:\n")
    print(REF_TEXT)
    print("\n" + "="*80)

    print("RAW ASR:\n")
    print(RAW_TEXT)
    print("\n" + "="*80)

    print("FIXED:\n")
    print(mapp_asr(RAW_TEXT))

REF:

Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным.

RAW ASR:

Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын ба

In [22]:
# ==============================
# METRICS (WER, CER, BLEU, chrF, TER, BERTScore)
# ==============================

import numpy as np
import re

# ---- NORMALIZATION ----
def norm_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---- WER ----
def wer(ref, hyp):
    # Тыныс белгісі мен әріп регистрі WER-ді жасанды түрде көтермеу үшін
    # салыстырудан бұрын екі мәтінді де нормализациялаймыз.
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    r = ref.split()
    h = hyp.split()

    d = np.zeros((len(r)+1, len(h)+1), dtype=int)

    for i in range(len(r)+1):
        d[i][0] = i
    for j in range(len(h)+1):
        d[0][j] = j

    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,      # deletion
                d[i][j-1] + 1,      # insertion
                d[i-1][j-1] + cost # substitution
            )

    return d[len(r)][len(h)] / max(1, len(r))

# ---- CER ----
def cer(ref, hyp):
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    r = list(ref)
    h = list(hyp)

    d = np.zeros((len(r)+1, len(h)+1), dtype=int)

    for i in range(len(r)+1):
        d[i][0] = i
    for j in range(len(h)+1):
        d[0][j] = j

    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,
                d[i][j-1] + 1,
                d[i-1][j-1] + cost
            )

    return d[len(r)][len(h)] / max(1, len(r))

# ---- BLEU ----
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def bleu_score(ref, hyp):
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref.split()], hyp.split(), smoothing_function=smoothie) * 100

# ---- chrF ----
from sacrebleu.metrics import CHRF

def chrf_score(ref, hyp):
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    chrf = CHRF()
    return chrf.sentence_score(hyp, [ref]).score

# ---- TER ----
from sacrebleu.metrics import TER

def ter_score(ref, hyp):
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    ter = TER()
    return ter.sentence_score(hyp, [ref]).score

# ---- BERTScore ----
from bert_score import score

def bertscore(ref, hyp):
    # BERTScore семантикалық ұқсастықты өлшейді, сондықтан пунктуацияға
    # WER/CER-ге қарағанда әлдеқайда аз сезімтал — бірақ бірізділік үшін
    # мұнда да нормализацияланған мәтінді қолданамыз.
    ref = norm_text(ref)
    hyp = norm_text(hyp)
    P, R, F1 = score([hyp], [ref], lang="kk")
    return float(F1.mean())

In [23]:
# =========================
# TEXTS
# =========================


REF_12 = "Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."

raw_text_12 = "Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай болыгерек? Талға төндір сайтасын, біздің балақ нағашылар мытасын. Ой, жандарым. Қайдағы нағашы, дән үшер болың өзімізге тапқан. Ахынмен солдай жоғпаттады болды. Жалын."

fixed_text_12 = "Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы? Қалай ол? Балам анау алайын деп жатқан қызың. Қазақ па әйтеуір кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек? Талғат сен дұрыс айтасың біздің балалар нағашыларына тартқан. Ой жандарым. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным."

In [24]:
# =========================
# NORMALIZATION
# =========================
def norm_text(t):
    return t.lower().strip()

# =========================
# PRINT TEXTS
# =========================
print("REF   :", REF_12)
print("RAW   :", raw_text_12)
print("FIXED :", fixed_text_12)

# =========================
# METRICS
# =========================
print("\nWER RAW :", wer(norm_text(REF_12), norm_text(raw_text_12)))
print("WER FIX :", wer(norm_text(REF_12), norm_text(fixed_text_12)))

print("\nCER RAW :", cer(REF_12, raw_text_12))
print("CER FIX :", cer(REF_12, fixed_text_12))

print("\nBLEU RAW :", bleu_score(REF_12, raw_text_12))
print("BLEU FIX :", bleu_score(REF_12, fixed_text_12))

print("\nchrF RAW :", chrf_score(REF_12, raw_text_12))
print("chrF FIX :", chrf_score(REF_12, fixed_text_12))

print("\nTER RAW :", ter_score(REF_12, raw_text_12))
print("TER FIX :", ter_score(REF_12, fixed_text_12))

print("\nBERT RAW :", bertscore(REF_12, raw_text_12))
print("BERT FIX :", bertscore(REF_12, fixed_text_12))

REF   : Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным.
RAW   : Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деген осын бай

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERT RAW : 0.76822829246521


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT FIX : 0.978399932384491


## Салыстыру: алгоритмдік MAPP v2 vs қолмен жазылған pipeline

Бұрын (cell 23) тек REF_12 мен қолмен (hardcoded PHRASE_REPLACEMENTS/
WORD_MAP) түзетілген fixed_text_12 салыстырылды. Төменде **тек
алгоритмдік** `mapp_asr_process` (жаңартылған MAPP v2) арқылы алынған
нәтиже де қосымша есептеледі — бұл 100 аудиоға генерализацияланатын
жалғыз нұсқа. Сонымен қатар `recoverability_report` осы бір мысал үшін
теориялық жоғарғы шекті көрсетеді.

In [25]:
algo_fixed_text_12 = mapp_asr_process(raw_text_12)

print("REF        :", REF_12)
print("RAW        :", raw_text_12)
print("ALGO FIXED :", algo_fixed_text_12)

print("\nWER RAW        :", wer(norm_text(REF_12), norm_text(raw_text_12)))
print("WER ALGO FIXED :", wer(norm_text(REF_12), norm_text(algo_fixed_text_12)))

print("\nCER RAW        :", cer(REF_12, raw_text_12))
print("CER ALGO FIXED :", cer(REF_12, algo_fixed_text_12))

print("\nBLEU RAW        :", bleu_score(REF_12, raw_text_12))
print("BLEU ALGO FIXED :", bleu_score(REF_12, algo_fixed_text_12))

print("\nchrF RAW        :", chrf_score(REF_12, raw_text_12))
print("chrF ALGO FIXED :", chrf_score(REF_12, algo_fixed_text_12))

print("\nTER RAW        :", ter_score(REF_12, raw_text_12))
print("TER ALGO FIXED :", ter_score(REF_12, algo_fixed_text_12))

print("\nBERT RAW        :", bertscore(REF_12, raw_text_12))
print("BERT ALGO FIXED :", bertscore(REF_12, algo_fixed_text_12))

print("\n--- Recoverability (теориялық жоғарғы шек) ---")
print(recoverability_report(raw_text_12.lower().split(), REF_12.lower().split(), levenshtein))

REF        : Үйленейін деп жатырсың ба? Үйленгені нес? Әй неге ақылдаспайсыңдар ай сендер? Тұра тұршы. Қалай ол? Балам анау алайын деп жатқан қызың қазақ па әйтеуір. Кәріс болып жүрмесін. Тыңдаңыздаршы айтайықта бізде. Сұңқат екеуміз бір жылдан бері ақша жинағанбыз. Ақша? Не үшін? Квартираға первоначалка. Енді біздің өзіміздің үйіміз болады. Ақырындап іздей берсеңіздер болады. Жандарым сол. Міне бала деген осындай болу керек. Талғат сен дұрыс айтасың. Біздің балалар нағашыларына тартқан. Жандарым сол. Қайдағы нағашы дәл осы жері өзімізге тартқан. Әкем менің сондай жомарт адам болған. Жаным.
RAW        : Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді біздің өзіміздің өйіміз болады. Ахында пізде версеніз. Ой, жандарым, сол. Мене, бала деге

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERT RAW        : 0.76822829246521


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT ALGO FIXED : 0.7684035301208496

--- Recoverability (теориялық жоғарғы шек) ---
{'total_compared': 83, 'already_correct': 0, 'recoverable_by_mapp': 13, 'needs_better_asr': 70, 'recoverable_share_of_errors': 0.1566265060240964}


## Batch: барлық 100 аудио (30s×30, 1m×25, 2m×20, 5m×15, 10m×10)

Файлдар мен REF мәтіндер орналасуы (Drive-та расталған):
```
dataset_wav16k/audio/<duration>/<stem>.wav
dataset_wav16k/references/<duration>/<stem>.txt
```
`<stem>` екеуінде де бірдей (мыс. `1m_001`), сондықтан атау бойынша
автоматты сәйкестендіруге болады.

⚠️ **Ескерту:** Whisper "medium" GPU-сыз (CPU) баяу жұмыс істейді, әсіресе
`10m`/`5m` ұзақ жазбаларда. Runtime → Change runtime type → GPU таңдаңыз,
әйтпесе 100 аудио бірнеше сағатқа созылуы мүмкін. Әр 10 файлдан кейін
нәтиже CSV-ге аралық сақталады — үзіліп қалса, соңғы сақталған жерден
жалғастыруға болады.

In [26]:
import os

DATASET_BASE = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k"
DURATIONS = ["30s", "1m", "2m", "5m", "10m"]


def collect_pairs(base, durations):
    pairs = []
    for d in durations:
        audio_dir = os.path.join(base, "audio", d)
        ref_dir = os.path.join(base, "references", d)
        if not os.path.isdir(audio_dir):
            print(f"[WARN] аудио қалтасы жоқ: {audio_dir}")
            continue
        for fname in sorted(os.listdir(audio_dir)):
            if not fname.lower().endswith(".wav"):
                continue
            stem = os.path.splitext(fname)[0]
            ref_path = os.path.join(ref_dir, stem + ".txt")
            if not os.path.exists(ref_path):
                print(f"[WARN] REF жоқ, өткізіп жібереміз: {ref_path}")
                continue
            pairs.append({
                "duration": d,
                "stem": stem,
                "audio_path": os.path.join(audio_dir, fname),
                "ref_path": ref_path,
            })
    return pairs


pairs = collect_pairs(DATASET_BASE, DURATIONS)
print("Табылған (audio, ref) жұп саны:", len(pairs))
for d in DURATIONS:
    n = sum(1 for p in pairs if p["duration"] == d)
    print(f"  {d}: {n}")

Табылған (audio, ref) жұп саны: 100
  30s: 30
  1m: 25
  2m: 20
  5m: 15
  10m: 10


In [27]:
!pip install -q tqdm

In [28]:
import time
import pandas as pd

from tqdm.auto import tqdm


def norm_for_metrics(t: str) -> str:
    return t.lower().strip()


OUT_CSV = os.path.join(DATASET_BASE, "mapp_v2_batch_results.csv")

# Егер бұрын үзіліп қалған нәтиже болса, соны жалғастырамыз
already_done = set()
results = []
if os.path.exists(OUT_CSV):
    prev = pd.read_csv(OUT_CSV)
    results = prev.to_dict("records")
    already_done = set(zip(prev["duration"], prev["stem"]))
    print(f"Бұрынғы нәтиже табылды: {len(results)} жол, соларды өткізіп жібереміз.")

start_time = time.time()

for item in tqdm(pairs, desc="MAPP-ASR batch"):
    key = (item["duration"], item["stem"])
    if key in already_done:
        continue

    try:
        with open(item["ref_path"], encoding="utf-8") as f:
            ref_text = f.read().strip()

        raw_text_i = transcribe_whisper_local(item["audio_path"])
        fixed_text_i = mapp_asr_process(raw_text_i)

        row = {
            "duration": item["duration"],
            "stem": item["stem"],
            "ref": ref_text,
            "raw": raw_text_i,
            "fixed": fixed_text_i,
            "wer_raw": wer(norm_for_metrics(ref_text), norm_for_metrics(raw_text_i)),
            "wer_fixed": wer(norm_for_metrics(ref_text), norm_for_metrics(fixed_text_i)),
            "cer_raw": cer(ref_text, raw_text_i),
            "cer_fixed": cer(ref_text, fixed_text_i),
            "bleu_raw": bleu_score(ref_text, raw_text_i),
            "bleu_fixed": bleu_score(ref_text, fixed_text_i),
            "chrf_raw": chrf_score(ref_text, raw_text_i),
            "chrf_fixed": chrf_score(ref_text, fixed_text_i),
            "ter_raw": ter_score(ref_text, raw_text_i),
            "ter_fixed": ter_score(ref_text, fixed_text_i),
            "bertscore_raw": bertscore(ref_text, raw_text_i),
            "bertscore_fixed": bertscore(ref_text, fixed_text_i),
        }
        results.append(row)
        already_done.add(key)

    except Exception as e:
        print(f"[ERROR] {item['duration']}/{item['stem']}: {e}")
        continue

    # аралық сақтау
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(OUT_CSV, index=False)

df = pd.DataFrame(results)
df.to_csv(OUT_CSV, index=False)

elapsed = time.time() - start_time
print(f"\nАяқталды. Өңделген файл саны: {len(df)} / {len(pairs)}")
print(f"Уақыт: {elapsed/60:.1f} мин")
print("Сақталды:", OUT_CSV)
df.head()

Бұрынғы нәтиже табылды: 100 жол, соларды өткізіп жібереміз.


MAPP-ASR batch:   0%|          | 0/100 [00:00<?, ?it/s]


Аяқталды. Өңделген файл саны: 100 / 100
Уақыт: 0.0 мин
Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mapp_v2_batch_results.csv


,duration,stem,ref,raw,fixed,wer_raw,wer_fixed,cer_raw,cer_fixed,bleu_raw,bleu_fixed,chrf_raw,chrf_fixed,ter_raw,ter_fixed,bertscore_raw,bertscore_fixed
0,30s,30s_001,"Түптеп келген кезде, бәрі мәдениеттен басталад...",Тіп-тип келгенгезде бәрі мәдінеттен басталатт...,Тіп - тип келгеншеде бәрі мәдениеттен балталат...,0.575342,0.547945,0.160000,0.184000,13.301490,12.282432,65.143004,62.388029,57.534247,54.794521,0.834425,0.836931
1,30s,30s_002,Мемлекеттің бекітіп берген стандарты бар. Сол ...,Мемлекеттің беңлеп бірген стандарты бар. Солы...,Мемлекеттің еңлер бірген стандарты бар. Солы с...,0.711111,0.733333,0.291892,0.308108,1.852266,1.818265,59.499217,57.536851,71.111111,73.333333,0.785603,0.775036
2,30s,30s_003,Бұл екі мың он сегізінші жылдардан басталған б...,Бұл 2018 жылдардың басталған болатын. Сол зам...,Бұл 2018 жылдардың басталған болатын. Сол зама...,0.766667,0.733333,0.288684,0.288684,2.609358,3.575523,56.439752,58.650810,76.666667,73.333333,0.797169,0.802775
3,30s,30s_004,Сіз негізі осы қаржы және несие мамандығы бойы...,"Сіз негізі ой, қаржы және несе, мамадыға бойы...","Сіз негізі ой, қаржы және несе, мамандыға бойы...",0.549020,0.549020,0.276657,0.282421,12.153873,12.153873,61.111709,58.807596,54.901961,54.901961,0.763337,0.763527
4,30s,30s_005,Қаржылық сауаттылық пен жеке қаржы туралы білі...,Қаржылық саватылықпен жеке қаржы тұралы білем...,"Қаржылық сауатылығпен жеке қаржы туралы біле, ...",0.836066,0.836066,0.473068,0.484778,1.290400,2.527217,43.980436,45.055702,83.606557,83.606557,0.726321,0.730768


In [29]:
metric_cols = [
    "wer_raw", "wer_fixed", "cer_raw", "cer_fixed",
    "bleu_raw", "bleu_fixed", "chrf_raw", "chrf_fixed",
    "ter_raw", "ter_fixed", "bertscore_raw", "bertscore_fixed",
]

# Excel (.xlsx / XML) кейбір басқару таңбаларын (control characters) қабылдамайды —
# Whisper кейде осындай "боп-бос" таңбалар шығарады. Жазбас бұрын тазалаймыз.
import re as _re
_ILLEGAL_XLSX_RE = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def _clean_for_excel(val):
    if isinstance(val, str):
        return _ILLEGAL_XLSX_RE.sub("", val)
    return val

df = df.applymap(_clean_for_excel)

print("=== Ұзақтық бойынша орташа мәндер ===")
summary_by_duration = df.groupby("duration")[metric_cols].mean().round(4)
display(summary_by_duration)

print("\n=== ЖАЛПЫ ОРТАША (барлық өңделген аудио) ===")
overall = df[metric_cols].mean().round(4)
print(overall)

# Excel-ге сақтау
OUT_XLSX = os.path.join(DATASET_BASE, "mapp_v2_batch_results.xlsx")
with pd.ExcelWriter(OUT_XLSX) as writer:
    df.to_excel(writer, sheet_name="per_file", index=False)
    summary_by_duration.to_excel(writer, sheet_name="summary_by_duration")
    overall.to_frame("overall_mean").to_excel(writer, sheet_name="overall")
print("\nExcel сақталды:", OUT_XLSX)

# Жиынтық recoverability (теориялық жоғарғы шек) — барлық сөздер бойынша
all_raw_words, all_ref_words = [], []
for _, r in df.iterrows():
    all_raw_words += str(r["raw"]).lower().split()
    all_ref_words += str(r["ref"]).lower().split()

print("\n=== Recoverability (100 аудио бойынша жиынтық) ===")
print(recoverability_report(all_raw_words, all_ref_words, levenshtein))

=== Ұзақтық бойынша орташа мәндер ===


/tmp/ipykernel_981/4033699705.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(_clean_for_excel)


,wer_raw,wer_fixed,cer_raw,cer_fixed,bleu_raw,bleu_fixed,chrf_raw,chrf_fixed,ter_raw,ter_fixed,bertscore_raw,bertscore_fixed
duration,,,,,,,,,,,,
10m,0.7802,0.7914,0.3881,0.4053,5.5963,5.8480,54.9310,53.5364,84.2704,84.5221,0.7695,0.7667
1m,0.7600,0.7658,0.3236,0.3390,5.5057,5.3873,54.6768,52.9864,75.7031,76.2717,0.7692,0.7638
2m,0.8009,0.8094,0.3809,0.3979,4.1967,3.9530,49.8781,48.4052,80.4057,81.0644,0.7577,0.7581
30s,0.7980,0.7975,0.4294,0.4430,3.7084,3.9450,43.5929,42.1506,79.7287,79.6083,0.7488,0.7416
5m,0.7739,0.7760,0.3867,0.4022,5.8579,5.7438,52.3342,50.8069,80.4213,80.2970,0.7805,0.7767



=== ЖАЛПЫ ОРТАША (барлық өңделген аудио) ===
wer_raw             0.7837
wer_fixed           0.7881
cer_raw             0.3827
cer_fixed           0.3981
bleu_raw            4.7666
bleu_fixed          4.7673
chrf_raw           50.0659
chrf_fixed         48.5475
ter_raw            79.4158
ter_fixed          79.6601
bertscore_raw       0.7625
bertscore_fixed     0.7582
dtype: float64

Excel сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mapp_v2_batch_results.xlsx

=== Recoverability (100 аудио бойынша жиынтық) ===
{'total_compared': 24319, 'already_correct': 22, 'recoverable_by_mapp': 1252, 'needs_better_asr': 23045, 'recoverable_share_of_errors': 0.05152899534922007}


## ASR деңгейін жақсарту: kazakh-whisper-large-v3-turbo (kk-turbo) + VAD

`MAPP_ASR_code`-тың README-індегі ЕҢ ЖАҚСЫ финалдық нәтиже (WER 0,3249)
дәл осы модельмен алынған: **`shyngys879/kazakh-whisper-large-v3-turbo`**
— `openai/whisper-large-v3-turbo`-ды қазақша (1500+ сағат) дообучен
нұсқа. Бұл `openai-whisper` кітапханасымен емес, **`transformers`-тің
`pipeline()`** арқылы жүктеледі (HF checkpoint).

Екі кесу тәсілін салыстырамыз:
- **chunk** — 30 секунд таймер бойынша (қарапайым)
- **VAD** — пауза бойынша (`crf_code/src/vad.py`), сөзді бұзбай кеседі

`vad.py`-дегі өлшем (басқа модельде): ұзақ режим WER 0,48 → VAD WER 0,29.


In [30]:
!pip -q install pydub

import torch
from transformers import pipeline

KK_TURBO_MODEL = "shyngys879/kazakh-whisper-large-v3-turbo"

_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
kk_asr = pipeline(
    "automatic-speech-recognition",
    model=KK_TURBO_MODEL,
    torch_dtype=_dtype,
    device=0 if torch.cuda.is_available() else -1,
)
print("Жүктелді:", KK_TURBO_MODEL, "| device:",
      "cuda" if torch.cuda.is_available() else "cpu")

KK_GEN_KWARGS = dict(language="kk", task="transcribe")

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Жүктелді: shyngys879/kazakh-whisper-large-v3-turbo | device: cuda


In [31]:
import os, shutil, tempfile
from pydub import AudioSegment

CHUNK_MS = 30_000  # 30 секунд — ұзын аудиода "тұйықталуды" болдырмау үшін


def _split_audio_chunks(audio_path, chunk_ms=CHUNK_MS):
    audio = AudioSegment.from_file(audio_path)
    total_ms = len(audio)
    tmp_dir = tempfile.mkdtemp(prefix="kkturbo_chunks_")
    paths = []
    for start in range(0, max(total_ms, 1), chunk_ms):
        seg = audio[start:start + chunk_ms]
        seg_path = os.path.join(tmp_dir, f"chunk_{start:08d}.wav")
        seg.export(seg_path, format="wav")
        paths.append(seg_path)
    return paths, tmp_dir


def transcribe_kkturbo_chunk(audio_path: str) -> str:
    chunk_paths, tmp_dir = _split_audio_chunks(audio_path)
    texts = []
    try:
        for cp in chunk_paths:
            res = kk_asr(cp, generate_kwargs=KK_GEN_KWARGS)
            texts.append(str(res["text"]).strip())
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    return " ".join(t for t in texts if t)


raw_text_12_chunk = transcribe_kkturbo_chunk(AUDIO_MP3)
print("kk-turbo + 30s-chunk RAW:", raw_text_12_chunk[:300])

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for

kk-turbo + 30s-chunk RAW: түптеп келген кезде бәрі мәдениеттен басталады тағы да тамақтану мәдениеті чекап жасау жылына бір рет дедіңіз бе жарты жылда бір рет жылына бір рет жеткір рет жылына бір рет чекап жасап организмде не болып жатыр тіпті еш жеріңіз ауырмаса да мен дәрігерге өмір бақи қаралмай ақ сау жүрмін дегендеріңіз


In [32]:
import numpy as np

def vad_segment(x, sr=16000, win=0.025, min_sil=0.30, max_len=18.0, min_len=1.0, pad=0.25):
    """crf_code/src/vad.py-мен бірдей алгоритм: әр аудионың өз деңгейінен
    бейімделген табалдырық, тек үнсіздіктің ортасынан кеседі."""
    n = int(win * sr)
    if len(x) < n * 2:
        return [(0, len(x))]

    frames = x[: len(x) // n * n].reshape(-1, n)
    e = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    floor = np.percentile(e, 10)
    speech = np.percentile(e, 75)
    thr = floor + 0.12 * max(speech - floor, 1e-6)
    sil = e < thr

    need = max(1, int(min_sil / win))
    cuts, run = [], 0
    for i, s in enumerate(sil):
        if s:
            run += 1
        else:
            if run >= need:
                cuts.append((i - run // 2) * n)
            run = 0

    bounds = sorted({0, len(x)} | {c for c in cuts if 0 < c < len(x)})
    segs, a = [], bounds[0]
    for b in bounds[1:]:
        if (b - a) / sr < min_len and b != bounds[-1]:
            continue
        while (b - a) / sr > max_len:
            segs.append((a, a + int(max_len * sr)))
            a += int(max_len * sr)
        segs.append((a, b))
        a = b

    return [(max(0, s - int(pad * sr)), min(len(x), e2 + int(pad * sr))) for s, e2 in segs]


def _load_audio_array(audio_path, target_sr=16000):
    """soundfile орнына pydub (ffmpeg) арқылы жүктейміз -- ол 'Format not
    recognised' қатесін шығаратын .wav контейнерлерін де түсінеді
    (chunk-функциясы да дәл осы жолмен жұмыс істейді)."""
    audio = AudioSegment.from_file(audio_path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    max_val = float(1 << (8 * audio.sample_width - 1))
    samples = samples / max_val
    return samples, target_sr


def transcribe_kkturbo_vad(audio_path: str, min_sil=0.30, max_len=18.0, pad=0.25) -> str:
    x, sr = _load_audio_array(audio_path, target_sr=16000)

    segs = vad_segment(x, sr, min_sil=min_sil, max_len=max_len, pad=pad)
    texts = []
    for s, e in segs:
        chunk = x[s:e]
        if len(chunk) < sr * 0.2:
            continue
        res = kk_asr({"array": chunk, "sampling_rate": sr}, generate_kwargs=KK_GEN_KWARGS)
        texts.append(str(res["text"]).strip())
    return " ".join(t for t in texts if t)


raw_text_12_vad = transcribe_kkturbo_vad(AUDIO_MP3)
print("kk-turbo + VAD RAW:", raw_text_12_vad[:300])
print()
print("Ескі (medium) RAW:", raw_text_12[:300] if "raw_text_12" in dir() else "(жоқ)")

kk-turbo + VAD RAW: түптеп келген кезде бәрі мәдениеттен басталады тағы да тамақтану мәдениеті чекап жасау жылына бір рет дедіңіз бе жарты жылда бір рет жылына бір рет жеткілді жылына бір рет чекап жасап организмде не болып жатыр тіпті еш жеріңіз ауырмаса да мен дәрігерге өмір бақи бақиқ қаралмай ақ сау жүрмін дегендер

Ескі (medium) RAW: Құрынен евріңсін ба? Құрын ғындысы я? Қын еге ақылды жүріп үшінді жүндір? Идіра дүрші? Қалай үңұр? Балам, ано, алай ен еврі қан қызың. Қазақ бай кеуір, әрсіз бай ұжырмесін. Түндәңіздар шайта қтамыз ді. Сұқатты кеумыз, Бір жылдан бейрі ақшы жүне ағымыз. Ақшы? Еші? Но, қвартираға перван айчалқы. Енді 


### 100 аудионы kk-turbo-мен екі тәсілмен (chunk / VAD) қайта өңдеу

Екеуі де бөлек CSV-ге сақталады, екеуі де resume-safe (әр 10 файлдан
кейін аралық сақтау).

In [33]:
def run_batch(transcribe_fn, out_name, desc):
    out_csv = os.path.join(DATASET_BASE, out_name)
    done, results = set(), []
    if os.path.exists(out_csv):
        prev = pd.read_csv(out_csv)
        results = prev.to_dict("records")
        done = set(zip(prev["duration"], prev["stem"]))
        print(f"{out_name}: бұрынғы нәтиже {len(results)} жол, өткізіп жібереміз.")

    t0 = time.time()
    for item in tqdm(pairs, desc=desc):
        key = (item["duration"], item["stem"])
        if key in done:
            continue
        try:
            with open(item["ref_path"], encoding="utf-8") as f:
                ref_text = f.read().strip()
            raw_text_i = transcribe_fn(item["audio_path"])
            fixed_text_i = mapp_asr_process(raw_text_i)
            row = {
                "duration": item["duration"], "stem": item["stem"],
                "ref": ref_text, "raw": raw_text_i, "fixed": fixed_text_i,
                "wer_raw": wer(norm_for_metrics(ref_text), norm_for_metrics(raw_text_i)),
                "wer_fixed": wer(norm_for_metrics(ref_text), norm_for_metrics(fixed_text_i)),
                "cer_raw": cer(ref_text, raw_text_i),
                "cer_fixed": cer(ref_text, fixed_text_i),
                "bleu_raw": bleu_score(ref_text, raw_text_i),
                "bleu_fixed": bleu_score(ref_text, fixed_text_i),
                "chrf_raw": chrf_score(ref_text, raw_text_i),
                "chrf_fixed": chrf_score(ref_text, fixed_text_i),
                "ter_raw": ter_score(ref_text, raw_text_i),
                "ter_fixed": ter_score(ref_text, fixed_text_i),
                "bertscore_raw": bertscore(ref_text, raw_text_i),
                "bertscore_fixed": bertscore(ref_text, fixed_text_i),
            }
            results.append(row)
            done.add(key)
        except Exception as e:
            print(f"[ERROR] {item['duration']}/{item['stem']}: {e}")
            continue
        if len(results) % 10 == 0:
            pd.DataFrame(results).to_csv(out_csv, index=False)

    pd.DataFrame(results).to_csv(out_csv, index=False)
    print(f"{out_name}: аяқталды, {len(results)}/{len(pairs)}, {(time.time()-t0)/60:.1f} мин")
    return out_csv


OUT_CSV_KKTURBO_CHUNK = run_batch(transcribe_kkturbo_chunk,
                                  "mapp_v2_batch_results_kkturbo_chunk.csv",
                                  "kk-turbo + 30s-chunk batch")

mapp_v2_batch_results_kkturbo_chunk.csv: бұрынғы нәтиже 100 жол, өткізіп жібереміз.


kk-turbo + 30s-chunk batch:   0%|          | 0/100 [00:00<?, ?it/s]

mapp_v2_batch_results_kkturbo_chunk.csv: аяқталды, 100/100, 0.0 мин


In [34]:
OUT_CSV_KKTURBO_VAD = run_batch(transcribe_kkturbo_vad,
                                "mapp_v2_batch_results_kkturbo_vad.csv",
                                "kk-turbo + VAD batch")

mapp_v2_batch_results_kkturbo_vad.csv: бұрынғы нәтиже 90 жол, өткізіп жібереміз.


kk-turbo + VAD batch:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mapp_v2_batch_results_kkturbo_vad.csv: аяқталды, 100/100, 91.3 мин


### Салыстыру: medium (ескі) vs kk-turbo+chunk vs kk-turbo+VAD

In [35]:
metric_cols3 = ["wer", "cer", "bleu", "chrf", "ter", "bertscore"]
OUT_CSV_OLD = os.path.join(DATASET_BASE, "mapp_v2_batch_results.csv")

df_old = pd.read_csv(OUT_CSV_OLD) if os.path.exists(OUT_CSV_OLD) else None
df_chunk = pd.read_csv(OUT_CSV_KKTURBO_CHUNK) if os.path.exists(OUT_CSV_KKTURBO_CHUNK) else None
df_vad = pd.read_csv(OUT_CSV_KKTURBO_VAD) if os.path.exists(OUT_CSV_KKTURBO_VAD) else None

rows = []
for m in metric_cols3:
    row = {"metric": m}
    if df_old is not None:
        row["medium_raw"] = df_old[f"{m}_raw"].mean()
    if df_chunk is not None:
        row["kkturbo_chunk_raw"] = df_chunk[f"{m}_raw"].mean()
    if df_vad is not None:
        row["kkturbo_vad_raw"] = df_vad[f"{m}_raw"].mean()
    rows.append(row)

kkturbo_comparison_df = pd.DataFrame(rows).round(4)
display(kkturbo_comparison_df)

import re as _re
_ILLEGAL_XLSX_RE = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def _clean_for_excel(val):
    return _ILLEGAL_XLSX_RE.sub("", val) if isinstance(val, str) else val

OUT_XLSX_KKTURBO = os.path.join(DATASET_BASE, "mapp_asr_kkturbo_vad_comparison.xlsx")
with pd.ExcelWriter(OUT_XLSX_KKTURBO) as writer:
    kkturbo_comparison_df.to_excel(writer, sheet_name="summary_comparison", index=False)
    if df_chunk is not None:
        df_chunk.applymap(_clean_for_excel).to_excel(writer, sheet_name="kkturbo_chunk_per_file", index=False)
    if df_vad is not None:
        df_vad.applymap(_clean_for_excel).to_excel(writer, sheet_name="kkturbo_vad_per_file", index=False)
print("\nExcel сақталды:", OUT_XLSX_KKTURBO)

,metric,medium_raw,kkturbo_chunk_raw,kkturbo_vad_raw
0,wer,0.7837,0.5468,0.7093
1,cer,0.3827,0.2891,0.4089
2,bleu,4.7666,29.5915,28.3368
3,chrf,50.0659,71.1574,74.8785
4,ter,79.4158,57.9746,76.6581
5,bertscore,0.7625,0.8154,0.8018


/tmp/ipykernel_981/2439816115.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_chunk.applymap(_clean_for_excel).to_excel(writer, sheet_name="kkturbo_chunk_per_file", index=False)
/tmp/ipykernel_981/2439816115.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_vad.applymap(_clean_for_excel).to_excel(writer, sheet_name="kkturbo_vad_per_file", index=False)



Excel сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mapp_asr_kkturbo_vad_comparison.xlsx
